# 🚀 IndicSent-M: Experiment 2A Training on Google Colab (GPU)

This notebook trains the **IndicSent-M** multimodal multilingual architecture for **Experiment 2A**:
- **Text Encoder:** MuRIL (`google/muril-base-cased`), trainable (lr: `1e-5`), with attention-masked mean pooling
- **Vision Encoder:** OpenAI CLIP ViT-B/32 (`openai/clip-vit-base-patch32`), frozen backbone (`freeze: true`, lr: `0.0`), genuine visual CLS/pooler extraction
- **Multimodal Fusion:** Cross-Modal Multi-Head Attention + Gated Multimodal Fusion Unit (fusion lr: `2e-5`, projection dim 512)
- **Multi-Task Heads:** 3-class Sentiment (Neg, Neu, Pos) + 2-class Offensive (Non-Offensive, Offensive)
- **Batch Size & Accumulation:** Micro-batch size 4, Gradient Accumulation 4 (Effective batch size 16)
- **Gradient Normalization:** Exact sample-weighted normalization over actual valid labels per accumulation window
- **Precision:** FP16 Automatic Mixed Precision (`torch.cuda.amp.autocast`) enabled for Tesla T4
- **Primary Selection Metric:** Language-balanced sentiment macro-F1: `selection_score = mean(hinglish_f1, bengali_f1)`
- **Output Directory:** Google Drive `/content/drive/MyDrive/IndicSentM/checkpoints/experiment_2a` (isolated from Run 1)


## Step 1: Verify GPU Acceleration
Ensure you have selected **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi

## Step 2: Mount Google Drive & Extract Project to Local SSD
> **Performance Note:** We extract the archive to Colab's local `/content/` NVMe SSD rather than reading images through Google Drive's FUSE mount to achieve maximum I/O throughput.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to your uploaded zip file in Drive (adjust folder name if different)
drive_zip_path = "/content/drive/MyDrive/IndicSentM/indicsent_project.zip"

# Fallback: If uploaded directly via Colab Files sidebar
if not os.path.exists(drive_zip_path):
    if os.path.exists("/content/indicsent_project.zip"):
        drive_zip_path = "/content/indicsent_project.zip"
    else:
        raise FileNotFoundError(f"Please upload indicsent_project.zip to Google Drive at {drive_zip_path} or directly in /content/")

# 3. Unzip directly into local SSD
print(f"Extracting {drive_zip_path} to /content/indicsent_project/...")
!unzip -q "{drive_zip_path}" -d /content/indicsent_project/

# 4. Change working directory
%cd /content/indicsent_project/
!ls -la

## Step 3: Install Required Dependencies

In [ ]:
!pip install -q transformers accelerate scikit-learn pillow pyyaml

## Step 4: Quick Smoke Test (Dry Run)
Runs 2 batches per split with Experiment 2A configuration to verify CUDA/T4 acceleration, forward pass, attention-masked pooling, loss calculation, and backward gradient computation without overwriting full checkpoints (saves to `checkpoints/experiment_2a/dry_run`).


In [ ]:
!python train.py --config configs/experiment_2.yaml --dry_run


## Step 5: Full Experiment 2A Multi-Task Training
Trains IndicSent-M with attention-masked pooling, genuine visual representation, frozen CLIP ViT-B/32 backbone, differential learning rates (MuRIL: `1e-5`, Fusion: `2e-5`), FP16 mixed precision, gradient accumulation (effective batch size 16), and language-balanced sentiment macro-F1 checkpoint selection.

All checkpoints write directly to Google Drive under `experiment_2a` so progress is never lost even if Colab disconnects, and Run 1 checkpoints remain completely untouched.


In [ ]:
!python train.py \
    --config configs/experiment_2.yaml \
    --output_dir /content/drive/MyDrive/IndicSentM/checkpoints/experiment_2a


## Step 5b: Resume Training After Interruption (If Colab Disconnects)
If your Colab session times out or disconnects mid-training, run this cell to automatically restore model weights, AdamW optimizer moments, scheduler position, and scaler states from the last completed epoch.


In [ ]:
!python train.py \
    --config configs/experiment_2.yaml \
    --output_dir /content/drive/MyDrive/IndicSentM/checkpoints/experiment_2a \
    --resume /content/drive/MyDrive/IndicSentM/checkpoints/experiment_2a/checkpoint_last.pt


## Step 6: Backup Verification & Checkpoint Inspection
Confirms all resumable and lightweight inference checkpoints are securely stored on Google Drive under `experiment_2a`.


In [ ]:
!ls -lh /content/drive/MyDrive/IndicSentM/checkpoints/experiment_2a/
print("✔ Checkpoints verified on persistent Google Drive storage.")
